# M4 · Métricas de transcripción


## 1. Clonar repositorio

In [13]:
RAMA = "LuisM4"

!git clone --branch {RAMA} --single-branch https://github.com/luisNP21/Topicos-IA.git /content/repo

fatal: destination path '/content/repo' already exists and is not an empty directory.


In [14]:
%cd /content/repo
!git pull
%cd M4/metricas_transcripcion
!pip install -r requirements.txt -q

/content/repo
Already up to date.
/content/repo/M4/metricas_transcripcion


## 2. Montar Drive y definir `PROJECT_ROOT`

In [15]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/TopicosIA/Proyecto-Salud")
with open("/content/repo/.env", "w") as f:
    f.write(f"PROJECT_ROOT={PROJECT_ROOT}\n")

manifest = PROJECT_ROOT / "M4/data/imagenes/manifest.json"
print("Manifiesto:", manifest, "| existe:", manifest.exists())
print("Caches de transcripcion disponibles:")
for p in sorted((PROJECT_ROOT / "M4/vision/predictions").glob("transcripciones_*.json")):
    print("  ", p.relative_to(PROJECT_ROOT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Manifiesto: /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M4/data/imagenes/manifest.json | existe: True
Caches de transcripcion disponibles:
   M4/vision/predictions/transcripciones_vlm_foto_realista.json
   M4/vision/predictions/transcripciones_vlm_limpio.json


## 3. Probar el módulo con datos simulados (opcional)

Genera transcripciones con errores conocidos y comprueba que las métricas los miden. **No son resultados del sistema**: van a `M4/outputs/metricas_transcripcion/simulado/`.

In [16]:
!python run_metricas.py --simular

[02:10:29] Simulacion: 6 niveles en /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M4/outputs/metricas_transcripcion/simulado/caches
[02:10:29] Evaluando /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M4/outputs/metricas_transcripcion/simulado/caches/transcripciones_simulado_limpio.json
[02:10:29] Evaluando /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M4/outputs/metricas_transcripcion/simulado/caches/transcripciones_simulado_foto_realista.json
[02:10:29] Evaluando /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M4/outputs/metricas_transcripcion/simulado/caches/transcripciones_simulado_manuscrito_foto.json
[02:10:29] Evaluando /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M4/outputs/metricas_transcripcion/simulado/caches/transcripciones_simulado_sombra_severo.json
[02:10:29] Evaluando /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M4/outputs/metricas_transcripcion/simulado/caches/transcripciones_simulado_rotacion_recorte_severo.json
[02:10:29] Evaluando /content/drive/MyDrive/To

## 4. Medir las transcripciones reales

Un cache de transcripciones por nivel (y, si existe, su cache NER en el mismo orden). Las rutas son relativas a `PROJECT_ROOT`. También se pueden fijar en `entradas` de `config.yaml`.

In [17]:
TRANSCRIPCIONES = [
    "M4/vision/predictions/transcripciones_vlm_limpio.json",
    "M4/vision/predictions/transcripciones_vlm_foto_realista.json",
]
NER = [
     "M2/predictions/vision_vlm_limpio.json",
     "M2/predictions/vision_vlm_foto_realista.json",
]

args = " ".join(f'"{t}"' for t in TRANSCRIPCIONES)
args_ner = (" --ner " + " ".join(f'"{n}"' for n in NER)) if NER else ""
if TRANSCRIPCIONES:
    !python run_metricas.py --transcripciones {args}{args_ner}
else:
    print("Agrega al menos un cache en TRANSCRIPCIONES")

[02:10:29] Evaluando M4/vision/predictions/transcripciones_vlm_limpio.json
[02:10:30] Evaluando M4/vision/predictions/transcripciones_vlm_foto_realista.json
[02:10:30] Listo en 0.6 s: /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M4/outputs/metricas_transcripcion
vlm          limpio                       CER_vis=0.0002 WER_vis=0.0014 lectura=1.0 inventadas=0.0
vlm          foto_realista                CER_vis=0.0029 WER_vis=0.0041 lectura=1.0 inventadas=0.0


## 5. Ver el resumen

In [18]:
from IPython.display import Markdown, display

for sub in ("", "simulado"):
    ruta = PROJECT_ROOT / "M4/outputs/metricas_transcripcion" / sub / "resumen.md"
    if ruta.exists():
        display(Markdown(ruta.read_text(encoding="utf-8")))

# Metricas de transcripcion (M4, tarea 5a)

Generado: 2026-10-26T02:10:30 · referencia principal: texto **visible** · manifiesto: `/content/drive/MyDrive/TopicosIA/Proyecto-Salud/M4/data/imagenes/manifest.json`

CER y WER micro (errores totales / tamano total de la referencia). `--` = no aplica.

| Sistema | Nivel | Docs | CER visible | WER visible | CER completo | CER sin tildes | CER limpio (mismos docs) | Entidades leidas | Inventadas | Mal leidas |
|---|---|---|---|---|---|---|---|---|---|---|
| vlm | limpio | 15/15 | 0.000 | 0.001 | 0.000 | 0.000 | -- | 100.0 % | 0.0 % | 0.0 % |
| vlm | foto_realista | 15/15 | 0.003 | 0.004 | 0.000 | 0.003 | 0.000 | 100.0 % | 0.0 % | 0.0 % |

**Entidades leidas:** entidades del gold visibles en la imagen que aparecen tal cual en la transcripcion (no depende del NER). **Inventadas / mal leidas:** fraccion de las entidades predichas por el NER que no estan en el texto original (ver auditoria.py).

## Origen de las entidades predichas

| Sistema | Nivel | Predichas | Acierto | FP del NER | Texto extra | Mal leida | Inventada | Fuera de transcripcion |
|---|---|---|---|---|---|---|---|---|
| vlm | limpio | 32 | 7 | 25 | 0 | 0 | 0 | 0 |
| vlm | foto_realista | 32 | 7 | 25 | 0 | 0 | 0 | 0 |

## Avisos

- vlm / limpio: {"documentos_fuera_del_manifiesto": 0, "paginas_faltantes": 0, "paginas_sobrantes": 0, "ner_por_pagina": true}
- vlm / foto_realista: {"documentos_fuera_del_manifiesto": 0, "paginas_faltantes": 0, "paginas_sobrantes": 0, "ner_por_pagina": true}


> **SIMULADO.** Estas cifras salen de transcripciones simuladas con errores inyectados.
> Sirven para validar el modulo, no son resultados del sistema.

# Metricas de transcripcion (M4, tarea 5a)

Generado: 2026-10-26T02:10:29 · referencia principal: texto **visible** · manifiesto: `/content/drive/MyDrive/TopicosIA/Proyecto-Salud/M4/data/imagenes/manifest.json`

CER y WER micro (errores totales / tamano total de la referencia). `--` = no aplica.

| Sistema | Nivel | Docs | CER visible | WER visible | CER completo | CER sin tildes | CER limpio (mismos docs) | Entidades leidas | Inventadas | Mal leidas |
|---|---|---|---|---|---|---|---|---|---|---|
| simulado | limpio | 15/15 | 0.145 | 0.183 | 0.145 | 0.139 | -- | 54.5 % | 36.6 % | 12.2 % |
| simulado | foto_realista | 15/15 | 0.195 | 0.437 | 0.197 | 0.189 | 0.145 | 45.5 % | 33.3 % | 15.4 % |
| simulado | manuscrito_foto | 15/15 | 0.175 | 0.408 | 0.175 | 0.170 | 0.145 | 72.7 % | 28.6 % | 7.1 % |
| simulado | sombra_severo | 15/15 | 0.257 | 0.571 | 0.356 | 0.252 | 0.145 | 55.6 % | 47.1 % | 8.8 % |
| simulado | rotacion_recorte_severo | 15/15 | 0.219 | 0.511 | 0.322 | 0.216 | 0.145 | 55.6 % | 35.3 % | 11.8 % |
| simulado | manuscrito_severo | 15/15 | 0.197 | 0.487 | 0.197 | 0.191 | 0.145 | 63.6 % | 30.9 % | 9.5 % |

**Entidades leidas:** entidades del gold visibles en la imagen que aparecen tal cual en la transcripcion (no depende del NER). **Inventadas / mal leidas:** fraccion de las entidades predichas por el NER que no estan en el texto original (ver auditoria.py).

## Origen de las entidades predichas

| Sistema | Nivel | Predichas | Acierto | FP del NER | Texto extra | Mal leida | Inventada | Fuera de transcripcion |
|---|---|---|---|---|---|---|---|---|
| simulado | limpio | 41 | 7 | 14 | 0 | 5 | 15 | 0 |
| simulado | foto_realista | 39 | 6 | 12 | 2 | 6 | 13 | 0 |
| simulado | manuscrito_foto | 42 | 9 | 15 | 3 | 3 | 12 | 0 |
| simulado | sombra_severo | 34 | 6 | 9 | 0 | 3 | 16 | 0 |
| simulado | rotacion_recorte_severo | 34 | 6 | 12 | 0 | 4 | 12 | 0 |
| simulado | manuscrito_severo | 42 | 6 | 18 | 1 | 4 | 13 | 0 |

## CER por dificultad de la caligrafia (niveles manuscritos)

| Sistema | Nivel | Dificultad | Docs | CER | WER |
|---|---|---|---|---|---|
| simulado | manuscrito_foto | dificil | 5 | 0.144 | 0.366 |
| simulado | manuscrito_foto | facil | 6 | 0.188 | 0.439 |
| simulado | manuscrito_foto | media | 4 | 0.201 | 0.424 |
| simulado | manuscrito_severo | dificil | 5 | 0.144 | 0.464 |
| simulado | manuscrito_severo | facil | 6 | 0.221 | 0.496 |
| simulado | manuscrito_severo | media | 4 | 0.234 | 0.505 |

La dificultad de cada caligrafia se etiqueto a ojo (ver M4/eval_visual/config.yaml).
